# Iris Dataset Classification Comparison
### Comparing Logistic Regression, Decision Tree, and Random Forest on the Exact Same Dataset

This notebook demonstrates:
1. Loading the Iris dataset
2. Splitting into train and test sets with reproducible random state and stratification
3. Training and evaluating Logistic Regression, Decision Tree, and Random Forest on the identical dataset partition.

In [ ]:
# Step 0: Install dependencies if running in Colab/Kaggle
# !pip install scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

## 1. Load Iris Data

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
X = iris.data
y = iris.target

print(f"Total samples: {len(X)}")
print(f"Feature names: {list(iris.feature_names)}")
print(f"Target classes: {list(iris.target_names)}")
df.head()

## 2. Split into Train and Test Dataset
We use `random_state=42` and `stratify=y` to ensure that class balance is preserved and reproducible.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"X_train shape: {X_train.shape} ({len(X_train)} samples)")
print(f"X_test shape:  {X_test.shape} ({len(X_test)} samples)")
print("\nTest set class breakdown:")
print(pd.Series(y_test).map(lambda c: iris.target_names[c]).value_counts())

## 3. Compare Logistic Regression, Decision Tree, and Random Forest
Training all 3 models on the **exact same** `(X_train, y_train)` and evaluating on `(X_test, y_test)`.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini",
        max_depth=3,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

results = []
confusion_matrices = {}

for name, model in models.items():
    # Train on exact same train data
    model.fit(X_train, y_train)
    
    # Evaluate on exact same test data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision (Macro)": precision_score(y_test, y_test_pred, average="macro"),
        "Recall (Macro)": recall_score(y_test, y_test_pred, average="macro"),
        "F1-Score (Macro)": f1_score(y_test, y_test_pred, average="macro"),
        "F1-Score (Weighted)": f1_score(y_test, y_test_pred, average="weighted"),
        "Overfitting Gap": abs(accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_test_pred))
    })
    confusion_matrices[name] = confusion_matrix(y_test, y_test_pred)

# Display results comparison table
results_df = pd.DataFrame(results)
results_df.style.highlight_max(subset=["Test Accuracy", "F1-Score (Macro)"], color="#c6f6d5")

## 4. Visualizing Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, cm) in zip(axes, confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    ax.set_title(f"{name} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()